In [232]:
import hashlib

def calculate_sha256(filepath):
    sha256_hash = hashlib.sha256()
    with open(filepath, "rb") as f:
        # อ่านทีละ 4096 bytes แทนการโหลดทั้งไฟล์เข้า memory
        for byte_block in iter(lambda: f.read(4096), b""):
            sha256_hash.update(byte_block)
    return sha256_hash.hexdigest()

files = ["secom.data", "secom_labels.data", "secom.names"]

registry = {}
for fname in files:
    file_hash = calculate_sha256(f"secom/{fname}")
    registry[fname] = file_hash
    print(f"{fname}: {file_hash}")

secom.data: 20f0e7ee434f7dcbae0eea9ffff009a2b57f42d6b0dc9a5bd4f00782c0a3374c
secom_labels.data: 126884cf453705c9e61a903fe906f0665a3b45ce3639e621edc5c93c89627e03
secom.names: 6d91b0b46cdee03064ee3e3112f937c1b3f7fcd9933575794ec07974e6f1ea59


01_secom_data_audit.py

In [233]:
import json
import os
from datetime import datetime

registry_path = "data_registry.json"

# ลบไฟล์เก่าถ้ามีอยู่แล้ว
if os.path.exists(registry_path):
    os.remove(registry_path)
    print(f"ลบไฟล์เก่า {registry_path} เรียบร้อย")

registry_record = {
    "dataset": "SECOM",
    "source": "UCI Machine Learning Repository (ID: 179)",
    "download_date": datetime.now().strftime("%Y-%m-%d"),
    "files": registry
}

with open(registry_path, "w") as f:
    json.dump(registry_record, f, indent=2)

print(f"สร้างไฟล์ {registry_path} ใหม่เรียบร้อย")
print(json.dumps(registry_record, indent=2))

ลบไฟล์เก่า data_registry.json เรียบร้อย
สร้างไฟล์ data_registry.json ใหม่เรียบร้อย
{
  "dataset": "SECOM",
  "source": "UCI Machine Learning Repository (ID: 179)",
  "download_date": "2026-09-05",
  "files": {
    "secom.data": "20f0e7ee434f7dcbae0eea9ffff009a2b57f42d6b0dc9a5bd4f00782c0a3374c",
    "secom_labels.data": "126884cf453705c9e61a903fe906f0665a3b45ce3639e621edc5c93c89627e03",
    "secom.names": "6d91b0b46cdee03064ee3e3112f937c1b3f7fcd9933575794ec07974e6f1ea59"
  }
}


In [234]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

# secom.data: whitespace-separated, ไม่มี header
X = pd.read_csv("secom/secom.data", sep=r"\s+", header=None)

# secom_labels.data: column 1 = label (+1/-1), column 2 = timestamp
y = pd.read_csv("secom/secom_labels.data", sep=r"\s+", header=None,
                           names=["label", "timestamp"])

print("X.shape:", X.shape)
print("y.shape:", y.shape)
print("the number of predictor variables:", X.shape[1])

X.shape: (1567, 590)
y.shape: (1567, 2)
the number of predictor variables: 590


In [235]:
display(y.head())

,label,timestamp
0,-1,19/07/2008 11:55:00
1,-1,19/07/2008 12:32:00
2,1,19/07/2008 13:17:00
3,-1,19/07/2008 14:43:00
4,-1,19/07/2008 15:22:00


In [236]:
display(X.head())

,0,1,2,3,4,5,6,7,8,9,...,580,581,582,583,584,585,586,587,588,589
0,3030.93,2564.00,2187.7333,1411.1265,1.3602,100.0,97.6133,0.1242,1.5005,0.0162,...,NaN,NaN,0.5005,0.0118,0.0035,2.3630,NaN,NaN,NaN,NaN
1,3095.78,2465.14,2230.4222,1463.6606,0.8294,100.0,102.3433,0.1247,1.4966,-0.0005,...,0.0060,208.2045,0.5019,0.0223,0.0055,4.4447,0.0096,0.0201,0.0060,208.2045
2,2932.61,2559.94,2186.4111,1698.0172,1.5102,100.0,95.4878,0.1241,1.4436,0.0041,...,0.0148,82.8602,0.4958,0.0157,0.0039,3.1745,0.0584,0.0484,0.0148,82.8602
3,2988.72,2479.90,2199.0333,909.7926,1.3204,100.0,104.2367,0.1217,1.4882,-0.0124,...,0.0044,73.8432,0.4990,0.0103,0.0025,2.0544,0.0202,0.0149,0.0044,73.8432
4,3032.24,2502.87,2233.3667,1326.5200,1.5334,100.0,100.3967,0.1235,1.5031,-0.0031,...,NaN,NaN,0.4800,0.4766,0.1045,99.3032,0.0202,0.0149,0.0044,73.8432


In [241]:

class_distribution = pd.DataFrame(y["label"].value_counts())
class_distribution["percentage"] = round((class_distribution["count"] / class_distribution["count"].sum()) * 100, 2)
class_distribution.reset_index(inplace=True)
class_distribution['label'] = class_distribution['label'].map({-1: 'Pass', 1: 'Fail'})
display(class_distribution)

,label,count,percentage
0,Pass,1463,93.36
1,Fail,104,6.64


In [238]:
def save_figure(fig, script_name, filename, base_dir="graph", dpi=150):
    target_dir = os.path.join(base_dir, script_name)
    
    # 1. ถ้า folder ไม่มี → สร้างใหม่ (รวม parent folder ด้วย)
    os.makedirs(target_dir, exist_ok=True)
    
    target_path = os.path.join(target_dir, filename)
    
    # 2. ถ้ามีไฟล์ชื่อเดิมอยู่แล้ว → ลบทิ้งก่อน
    if os.path.exists(target_path):
        os.remove(target_path)
    
    # 3. save ไฟล์ใหม่
    fig.savefig(target_path, dpi=dpi, bbox_inches="tight")

In [239]:
colors = {"Pass": "#4C72B0", "Fail": "#C44E52"}
bar_colors = class_distribution["label"].map(colors)
 
fig, ax = plt.subplots(figsize=(6, 5))
bars = ax.bar(
    class_distribution["label"],
    class_distribution["count"],
    color=bar_colors,
    width=0.6,
)
 
ax.set_title("SECOM Class Distribution: Pass vs Fail", fontsize=13)
ax.set_ylabel("Observation Count")
ax.set_ylim(0, class_distribution["count"].max() * 1.15)
 
# Label on top of each bar: use count AND percentage columns directly
for bar, count, pct in zip(bars, class_distribution["count"], class_distribution["percentage"]):
    ax.annotate(
        f"{count}\n({pct}%)",
        (bar.get_x() + bar.get_width() / 2, bar.get_height()),
        ha="center", va="bottom",
        fontsize=11, fontweight="bold",
    )
 
fig.tight_layout()
save_figure(fig, "01_secom_data_audit.py", "class_distribution.png", dpi=150)
plt.close(fig)
 
print("Saved -> outputs/fig_class_distribution_from_table.png")
print(class_distribution)

KeyError: 'label'

In [ ]:
total_missing = X.isnull().sum().sum()
total_missing
total_cells = X.shape[0] * X.shape[1]
missing_percentage = (total_missing / total_cells) * 100
print(f"Total missing values: {total_missing}")
print(f"Percentage of missing values: {missing_percentage:.2f}%")

Total missing values: 41951
Percentage of missing values: 4.54%


In [ ]:
missing_rate_per_feature = X.isnull().mean() * 100
missing_rate_per_feature

0      0.382897
1      0.446713
2      0.893427
3      0.893427
4      0.893427
         ...   
585    0.063816
586    0.063816
587    0.063816
588    0.063816
589    0.063816
Length: 590, dtype: float64

In [ ]:
bins = [0, 10, 20, 30, 40, 50, 60, 70, 80, 90, 100]
distribution_of_missing_rate = pd.DataFrame(pd.cut(missing_rate_per_feature, bins=bins).value_counts())
distribution_of_missing_rate.reset_index(inplace=True)
distribution_of_missing_rate.rename(columns={"index": "Missing Rate (%)","count": "Number of Features"}, inplace=True)
distribution_of_missing_rate.sort_values(by='Missing Rate (%)', inplace=True)
distribution_of_missing_rate["Missing Rate (%)"] = distribution_of_missing_rate["Missing Rate (%)"].astype(str).str.replace({"(": "", "]": "%", ", ": "-"})
distribution_of_missing_rate

,Missing Rate (%),Number of Features
0,0-10%,486
1,10-20%,20
7,20-30%,0
8,30-40%,0
3,40-50%,4
4,50-60%,4
2,60-70%,16
9,70-80%,0
5,80-90%,4
6,90-100%,4


In [ ]:
y['timestamp'] = pd.to_datetime(y['timestamp'])
print("Earliest Timestamp:", y['timestamp'].min())
print("Latest Timestamp:", y['timestamp'].max())
print("Number of Unique Timestamps:", y['timestamp'].nunique())
grouped_by_timestamp = y.groupby('timestamp').count().reset_index()
grouped_by_timestamp.rename(columns={'label': 'count'}, inplace=True)

Earliest Timestamp: 2008-07-19 11:55:00
Latest Timestamp: 2008-10-17 06:07:00
Number of Unique Timestamps: 1534


/var/folders/3q/m3mhwsps6nb6l4bnhg9cxrx40000gn/T/ipykernel_71279/2609356693.py:1: UserWarning: Parsing dates in %d/%m/%Y %H:%M:%S format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  y['timestamp'] = pd.to_datetime(y['timestamp'])


In [ ]:
import matplotlib.dates as mdates

y['date'] = y['timestamp'].dt.date   # ตัดเวลาออก เหลือแค่ วัน-เดือน-ปี

grouped_by_date = y.groupby('date').count().reset_index()
grouped_by_date.rename(columns={'label': 'count'}, inplace=True)
grouped_by_date['date'] = pd.to_datetime(grouped_by_date['date'])

plt.rcParams.update({
    "font.size": 11,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.edgecolor": "#333333",
    "figure.facecolor": "white",
})

fig, ax = plt.subplots(figsize=(12, 4.5))
ax.plot(grouped_by_date['date'], grouped_by_date['count'],
        color="#4C72B0", linewidth=1, marker="o", markersize=3)

ax.set_title("SECOM: Observation Count over Time (by day)", fontsize=13, fontweight="bold")
ax.set_ylabel("Number of Observations")
ax.set_xlabel("")
ax.xaxis.set_major_locator(mdates.WeekdayLocator(interval=2))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%d %b"))
ax.grid(axis="y", alpha=0.25)

for label in ax.get_xticklabels():
    label.set_rotation(40)
    label.set_ha("right")

fig.tight_layout()

save_figure(fig, script_name="02_temporal_audit", filename="observation_count_by_date.png")
plt.close(fig)


In [ ]:
# total_missing = X.isnull().sum().sum()
# total_missing
# total_cell = X.shape[0] * X.shape[1]
# missing_percentage = (total_missing / total_cell) * 100

# missing_rate_per_feature = X.isnull().mean() * 100
# bins = [0, 10, 20, 30, 40, 50, 60, 70, 80, 90, 100]
# distribution_of_missing_rate = pd.DataFrame(pd.cut(missing_rate_per_feature, bins=bins).value_counts())
# distribution_of_missing_rate.reset_index(inplace=True)
# distribution_of_missing_rate.rename(columns={"index": "Missing Rate (%)","count": "Number of Features"}, inplace=True)
# distribution_of_missing_rate.sort_values(by='Missing Rate (%)', inplace=True)
# distribution_of_missing_rate["Missing Rate (%)"] = distribution_of_missing_rate["Missing Rate (%)"].astype(str).str.replace({"(": "", "]": "%", ", ": "-"})

# print(f"Total missing values: {total_missing}")
# print(f"Percentage of missing values: {missing_percentage:.2f}%")
# print(distribution_of_missing_rate)

In [ ]:
import pandas as pd
import numpy as np
import os
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

def save_figure(fig, script_name, filename, base_dir="graph", dpi=150):
    target_dir = os.path.join(base_dir, script_name)
    os.makedirs(target_dir, exist_ok=True)
    
    target_path = os.path.join(target_dir, filename)
    
    if os.path.exists(target_path):
        os.remove(target_path)
    
    fig.savefig(target_path, dpi=dpi, bbox_inches="tight")

X = pd.read_csv("secom/secom.data", sep=r"\s+", header=None)

y = pd.read_csv("secom/secom_labels.data", sep=r"\s+", header=None,
                           names=["label", "timestamp"])

y['timestamp'] = pd.to_datetime(y['timestamp'])
y['label'] = y['label'].map({-1: 0, 1: 1})
print("Earliest Timestamp:", y['timestamp'].min())
print("Latest Timestamp:", y['timestamp'].max())
print("Number of Unique Timestamps:", y['timestamp'].nunique())

y['date'] = y['timestamp'].dt.date

grouped_by_date = y.groupby('date').agg({'label': ['count', 'sum', 'mean']}).reset_index()
grouped_by_date['date'] = pd.to_datetime(grouped_by_date['date'])

grouped_by_date.reset_index(inplace=True)


Earliest Timestamp: 2008-07-19 11:55:00
Latest Timestamp: 2008-10-17 06:07:00
Number of Unique Timestamps: 1534


/var/folders/3q/m3mhwsps6nb6l4bnhg9cxrx40000gn/T/ipykernel_71279/3960990329.py:24: UserWarning: Parsing dates in %d/%m/%Y %H:%M:%S format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  y['timestamp'] = pd.to_datetime(y['timestamp'])


MultiIndex([('index',      ''),
            ( 'date',      ''),
            ('label', 'count'),
            ('label',   'sum'),
            ('label',  'mean')],
           )

In [ ]:
X.var()

0        5420.167484
1        6465.399020
2         871.026144
3      195091.505178
4        3175.946899
           ...      
585        12.802321
586         0.000153
587         0.000078
588         0.000008
589      8815.692532
Length: 590, dtype: float64